In [ ]:
import os
import gc
import torch

# 🧹 1. 메모리 강제 청소 (GCP 서버 방어용)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
import os
import gc
import torch
import pandas as pd
import shutil
import time
from typing import List
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. 경로 설정 (GCP 서버 환경)
RAW_DATA_PATH = "/home/shared/"          # 원본 데이터가 있는 공용 폴더
MY_WORK_DIR = "/home/spai0721/Bidcoin"  # 내 작업 결과물이 저장될 곳
PROCESSED_CSV = os.path.join(MY_WORK_DIR, "processed_data.csv")
FAISS_INDEX_DIR = os.path.join(MY_WORK_DIR, "faiss_index")

# 폴더 생성
os.makedirs(MY_WORK_DIR, exist_ok=True)

# 2. API 및 모델 설정
os.environ["OPENAI_API_KEY"] = "sk"
EMBED_MODEL_NAME = "text-embedding-3-small"
RERANK_MODEL_NAME = "BAAI/bge-reranker-v2-m3" # 혹은 "Dongjin-kr/ko-reranker"

print(f"✅ 설정 완료: 결과물은 {MY_WORK_DIR}에 저장됩니다.")

In [ ]:
import subprocess
import os
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def run_preprocessing():
    SHARED_ROOT = "/home/shared/"
    DOCS_DIR = os.path.join(SHARED_ROOT, "files")
    
    print(f"🚀 [Step 1] HWP 포함 전방위 문서 전처리 시작...")
    all_docs = []

    # --- 1. 루트 CSV 로드 ---
    for f in os.listdir(SHARED_ROOT):
        if f.lower().endswith('.csv'):
            try:
                all_docs.extend(CSVLoader(os.path.join(SHARED_ROOT, f)).load())
                print(f"  ✅ 원본 CSV 로드: {f}")
            except: pass

    # --- 2. files/ 폴더 내 PDF & HWP 로드 ---
    if os.path.exists(DOCS_DIR):
        for f in os.listdir(DOCS_DIR):
            path = os.path.join(DOCS_DIR, f)
            ext = f.split('.')[-1].lower()
            
            try:
                if ext == 'pdf':
                    all_docs.extend(PyPDFLoader(path).load())
                    print(f"  ✅ PDF 로드: {f}")
                
                elif ext == 'hwp':
                    # 리눅스에서 HWP 텍스트 추출 (hwp5txt 사용)
                    # 결과물을 텍스트로 받아 LangChain Document 형식으로 변환합니다.
                    result = subprocess.run(['hwp5txt', path], capture_output=True, text=True)
                    if result.returncode == 0:
                        from langchain_core.documents import Document
                        all_docs.append(Document(page_content=result.stdout, metadata={"source": f}))
                        print(f"  ✅ HWP 로드: {f}")
                    else:
                        print(f"  ❌ HWP 로드 실패(에러): {f}")
            except Exception as e:
                print(f"  ❌ {f} 처리 중 오류: {e}")

    # --- 3. 동적 청킹 및 저장 ---
    if not all_docs:
        print("🛑 문서가 없습니다. 경로를 다시 확인하세요.")
        return

    print(f"✂️ 총 {len(all_docs)}개 파일/페이지 청킹 중...")
    # 입찰 서류는 표가 많으므로 청크를 너무 작게 나누지 않는 것이 유리함 (750자 추천)
    splitter = RecursiveCharacterTextSplitter(chunk_size=750, chunk_overlap=150)
    
    chunks = []
    for doc in all_docs:
        for text in splitter.split_text(doc.page_content):
            chunks.append({
                "content": text, 
                "source": os.path.basename(doc.metadata.get("source", "unknown"))
            })
    
    pd.DataFrame(chunks).to_csv(PROCESSED_CSV, index=False, encoding='utf-8-sig')
    print(f"✨ 완료: {len(chunks)}개 청크 저장됨 -> {PROCESSED_CSV}")

run_preprocessing()

In [ ]:
def build_vector_db():
    print(f"🧠 [Step 2] {EMBED_MODEL_NAME} 임베딩 및 FAISS 구축 시작...")
    df = pd.read_csv(PROCESSED_CSV)
    docs = [Document(page_content=str(row['content']), metadata={"source": row['source']}) for _, row in df.iterrows()]
    
    embeddings = OpenAIEmbeddings(model=EMBED_MODEL_NAME)
    vectorstore = FAISS.from_documents(docs, embeddings)
    
    # 인덱스 로컬 저장 (나중에 불러오기 위해)
    vectorstore.save_local(FAISS_INDEX_DIR)
    print(f"✅ 벡터 DB 저장 완료: {FAISS_INDEX_DIR}")

build_vector_db()

In [ ]:
def run_experiment(query):
    # 1. 1차 검색 (FAISS)
    embeddings = OpenAIEmbeddings(model=EMBED_MODEL_NAME)
    vectorstore = FAISS.load_local(FAISS_INDEX_DIR, embeddings, allow_dangerous_deserialization=True)
    candidates = vectorstore.similarity_search(query, k=15)
    
    # 2. 2차 재정렬 (Reranker) - 로컬 메모리 관리 포함
    print(f"⚖️ '{query}' 에 대한 재정렬 수행 중...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 모델 로드 (name 'model' is not defined 에러 방지를 위해 함수 내에서 명시적 선언)
    tokenizer = AutoTokenizer.from_pretrained(RERANK_MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(RERANK_MODEL_NAME).to(device)
    model.eval()

    scored_docs = []
    for doc in candidates:
        inputs = tokenizer(query, doc.page_content, return_tensors='pt', truncation=True, max_length=512).to(device)
        with torch.no_grad():
            score = model(**inputs).logits[0][0].item()
        scored_docs.append((score, doc))

    # 점수 정렬 및 출력
    scored_docs.sort(key=lambda x: x[0], reverse=True)
    
    print("\n" + "="*60)
    print(f"🎯 최종 TOP 3 결과 (BGE-Reranker 점수순)")
    print("="*60)
    for i, (score, doc) in enumerate(scored_docs[:3]):
        print(f"[{i+1}위 | Score: {score:.4f}] 출처: {doc.metadata['source']}")
        print(f"내용: {doc.page_content[:500]}...\n")

    # 3. 메모리 정리 (GCP 서버 RAM 확보용)
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# --- 여기서 테스트하고 싶은 질문을 입력하세요 ---
test_query = "우리 고객사가 교육 콘텐츠 구축 경험이 많은데, 추천할 만한 공고 있어?"
run_experiment(test_query)